# Introducción a la Ciencia de Datos: Tarea 2

Este notebook contiene la resolución de la Tarea 2. Es la continuación de la Tarea 1, reutilizando los mismos datos y la función `clean_text()` desarrollada en dicha tarea.

## Cargar bibliotecas (dependencias)

In [ ]:
from time import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

## Descarga del dataset
Se utilizan los mismos datos que en la Tarea 1.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train", cache_dir="../data")
df = ds.to_pandas()
print(f"Dataset cargado: {df.shape[0]:,} artículos, {df.shape[1]} columnas")

# Parte 1: Dataset y representación numérica de texto

## 1. Preparación del dataset

Se utilizará un conjunto de datos reducido de los **tres medios de prensa con mayor cantidad de artículos**.
Se reutiliza la función `clean_text()` de la Tarea 1, que combina título y artículo, convierte a minúsculas, elimina URLs y signos de puntuación.

El split train/test es **70/30 con muestreo estratificado**, lo que garantiza que la proporción de cada medio sea la misma en ambos conjuntos. Se fija `random_state=42` para reproducibilidad.

In [ ]:
def clean_text(df, col_title, col_article):
    """
    Combina título y artículo, y normaliza el texto:
    - Elimina dateline del artículo (primera línea hasta '\n')
    - Concatena title + article
    - Convierte a minúsculas
    - Elimina URLs
    - Elimina puntuación (conserva solo letras, números y espacios)
    - Colapsa espacios múltiples
    """
    # Eliminar dateline ANTES de combinar con el título
    article = df[col_article].fillna("").str.replace(r"^[^\n]*\n", "", regex=True)

    # Combinar título y artículo
    result = df[col_title].fillna("") + " " + article

    # Minúsculas
    result = result.str.lower()

    # Eliminar URLs
    result = result.str.replace(r"https?://\S+|www\.\S+", " ", regex=True)

    # Eliminar puntuación y caracteres especiales
    result = result.str.replace(r"[^a-z0-9\s]", " ", regex=True)

    # Colapsar espacios múltiples
    result = result.str.replace(r"\s+", " ", regex=True).str.strip()

    return result

In [ ]:
# Obtener los tres medios con mayor cantidad de artículos
top_3_publications = df["publication"].value_counts().head(3).index.tolist()
print("Top 3 medios:", top_3_publications)
print()
print(df["publication"].value_counts().head(3))

In [ ]:
# Filtrar el DataFrame para quedarse solo con los top 3 medios
df_top_3 = df[df["publication"].isin(top_3_publications)].copy()
print(f"Artículos en top 3: {len(df_top_3):,}")

# Aplicar clean_text sobre la combinación título + artículo
df_top_3["CleanText"] = clean_text(df_top_3, "title", "article")

# Verificar resultado
df_top_3[["title", "article", "CleanText"]].head(2)

In [ ]:
# Definir features (X) y etiquetas (y)
X = df_top_3["CleanText"]
y = df_top_3["publication"]

# Split 70/30 estratificado
# stratify=y garantiza que la proporción de cada medio sea la misma en train y test
# random_state=42 asegura reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print(f"Train: {len(X_train):,} artículos")
print(f"Test:  {len(X_test):,} artículos")

## 2. Verificación del balance de clases

Se verifica que la proporción de artículos de cada medio es similar en train y test, resultado directo del muestreo estratificado.

In [ ]:
# Calcular proporciones en train y test
train_dist = y_train.value_counts(normalize=True).rename("Train")
test_dist  = y_test.value_counts(normalize=True).rename("Test")
dist = pd.concat([train_dist, test_dist], axis=1) * 100

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(dist))
width = 0.35

bars1 = ax.bar(x - width/2, dist["Train"], width, label="Train", color="#4C72B0")
bars2 = ax.bar(x + width/2, dist["Test"],  width, label="Test",  color="#DD8452")

ax.set_xticks(x)
ax.set_xticklabels(dist.index, rotation=15, ha="right")
ax.set_ylabel("Proporción (%)")
ax.set_title("Balance de clases en Train vs Test")
ax.legend()
ax.set_ylim(0, 60)

# Etiquetas de valor
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("p1_2_balance_clases.png", dpi=150)
plt.show()
print("\nProporciones exactas:")
print(dist.round(1))

## 3. Representación Bag of Words

**¿Cómo funciona?**
Bag of Words (BoW) convierte texto en vectores numéricos contando la cantidad de veces que aparece cada palabra (token) del vocabulario en cada documento. El vocabulario se construye a partir del conjunto de entrenamiento. El resultado es una **matriz documento-término** donde:
- Cada **fila** representa un artículo.
- Cada **columna** representa una palabra del vocabulario.
- Cada **valor** es la cantidad de veces que esa palabra aparece en ese artículo.

**¿Por qué es una matriz sparse?**
El vocabulario puede tener decenas de miles de palabras, pero cada artículo usa solo unos cientos. La mayoría de las celdas son 0. Una matriz sparse almacena únicamente los valores no-nulos junto con sus coordenadas (fila, columna), ahorrando enormes cantidades de memoria RAM.

Por ejemplo, con 10.000 artículos y un vocabulario de 50.000 palabras, la matriz completa tendría 500 millones de celdas. Si solo el 0,5% son no-nulas, una representación densa ocuparía ~4 GB, mientras que la sparse ocupa apenas ~20 MB.

In [ ]:
# Transformar el texto de entrenamiento a Bag of Words
# Se ajusta (fit) SOLO sobre el train para no contaminar con información del test
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow  = bow_vectorizer.transform(X_test)

print(f"Tamaño de la matriz BoW de train: {X_train_bow.shape}")
print(f"  → {X_train_bow.shape[0]:,} artículos × {X_train_bow.shape[1]:,} palabras en el vocabulario")
print(f"  → Tipo: {type(X_train_bow)}")
print(f"  → Valores no-nulos: {X_train_bow.nnz:,} ({100*X_train_bow.nnz/X_train_bow.shape[0]/X_train_bow.shape[1]:.2f}% de celdas ocupadas)")

In [ ]:
# Ejemplo: primeras 10 palabras del vocabulario y su conteo en el primer artículo
vocab = bow_vectorizer.get_feature_names_out()
first_doc = X_train_bow[0]

print("Texto limpio (primeros 200 chars):")
print(X_train.iloc[0][:200])
print()

# Palabras con conteo > 0 en el primer artículo
doc_array = first_doc.toarray()[0]
nonzero_indices = doc_array.nonzero()[0]
print(f"Palabras con conteo > 0 en el primer artículo ({len(nonzero_indices)} palabras únicas):")
ejemplos = [(vocab[i], doc_array[i]) for i in nonzero_indices[:15]]
for palabra, conteo in sorted(ejemplos, key=lambda x: -x[1]):
    print(f"  '{palabra}': {int(conteo)}")

## 4. Representación TF-IDF

**¿Qué es un n-grama?**
Un n-grama es una secuencia de N palabras consecutivas. Con `ngram_range=(1,2)` el vectorizador considera tanto palabras individuales (unigramas) como pares de palabras adyacentes (bigramas). Por ejemplo, el texto "nueva york city" generaría los unigramas `nueva`, `york`, `city` y los bigramas `nueva york`, `york city`. Los bigramas capturan contexto y relaciones entre palabras que los unigramas ignoran.

**¿Qué es TF-IDF?**
TF-IDF mejora BoW penalizando palabras muy frecuentes en todos los documentos (que aportan poco información) y premiando palabras que son específicas de ciertos documentos:

- **TF (Term Frequency):** frecuencia de la palabra en el documento actual. Palabras que aparecen mucho en un artículo tienen mayor peso.
- **IDF (Inverse Document Frequency):** logaritmo inverso de la proporción de documentos que contienen esa palabra. Si una palabra aparece en todos los artículos (como "the", "a"), su IDF es cercano a 0. Si aparece en pocos documentos, su IDF es alto.
- **TF-IDF = TF × IDF**: el peso final premia palabras frecuentes en un documento pero raras en el corpus.

In [ ]:
# Obtener la representación TF-IDF del conjunto de entrenamiento
# Se ajusta SOLO sobre el train
tfidf_vectorizer = TfidfVectorizer(stop_words="english", use_idf=True)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print(f"Tamaño de la matriz TF-IDF de train: {X_train_tfidf.shape}")
print(f"  → {X_train_tfidf.shape[0]:,} artículos × {X_train_tfidf.shape[1]:,} términos en el vocabulario")
print(f"  → Tipo: {type(X_train_tfidf)}")
print(f"  → Valores no-nulos: {X_train_tfidf.nnz:,}")

## 5. Visualización SVD sobre TF-IDF

Se aplica PCA con 2 componentes para proyectar los vectores TF-IDF (de alta dimensión) en un espacio 2D visualizable. Se comparan distintas configuraciones del vectorizador para analizar cómo afectan la separabilidad entre medios.

In [ ]:
def plot_svd(X_matrix, y_labels, title, ax):
    """
    Aplica TruncatedSVD (equivalente a PCA para matrices sparse) con 2 componentes
    y grafica los puntos coloreados por medio de prensa.
    A diferencia de PCA, TruncatedSVD opera directamente sobre matrices sparse
    sin necesidad de convertirlas a densas, evitando el crash de RAM.
    """
    svd = TruncatedSVD(n_components=2, random_state=42)
    coords = svd.fit_transform(X_matrix)  # funciona directo con sparse

    df_plot = pd.DataFrame({
        "PC1": coords[:, 0],
        "PC2": coords[:, 1],
        "Medio": y_labels.values
    })

    palette = sns.color_palette("tab10", n_colors=df_plot["Medio"].nunique())
    for i, medio in enumerate(df_plot["Medio"].unique()):
        mask = df_plot["Medio"] == medio
        ax.scatter(df_plot.loc[mask, "PC1"], df_plot.loc[mask, "PC2"],
                   label=medio, alpha=0.4, s=8, color=palette[i])

    var = svd.explained_variance_ratio_
    ax.set_xlabel(f"SVD1 ({var[0]*100:.1f}% var. explicada)")
    ax.set_ylabel(f"SVD2 ({var[1]*100:.1f}% var. explicada)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, markerscale=2)
    return svd

In [ ]:
# Comparar 4 configuraciones de TfidfVectorizer usando TruncatedSVD para visualización 2D
# TruncatedSVD opera sobre matrices sparse directamente — sin crash de RAM
configs = [
    ("Base (sin stop_words, sin IDF)", TfidfVectorizer(use_idf=False)),
    ("Con stop_words='english'",        TfidfVectorizer(stop_words="english", use_idf=True)),
    ("Con use_idf=True",                TfidfVectorizer(use_idf=True)),
    ("Con ngram_range=(1,2)",           TfidfVectorizer(stop_words="english", use_idf=True, ngram_range=(1, 2))),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (title, vec) in zip(axes, configs):
    X_tr = vec.fit_transform(X_train)
    plot_svd(X_tr, y_train, title, ax)

plt.suptitle("SVD 2D sobre TF-IDF — comparación de configuraciones", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("p1_5_pca_configs.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Varianza explicada acumulada con hasta 10 componentes
# Reutilizamos X_train_tfidf (stop_words + use_idf) ya calculada en sección 1.4
# TruncatedSVD opera sobre la matriz sparse directamente — sin .toarray()
svd_10 = TruncatedSVD(n_components=10, random_state=42)
svd_10.fit(X_train_tfidf)

explained = svd_10.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(1, 11), explained * 100, color="#4C72B0")
ax1.set_xlabel("Componente SVD")
ax1.set_ylabel("Varianza explicada (%)")
ax1.set_title("Varianza explicada por componente")
ax1.set_xticks(range(1, 11))

ax2.plot(range(1, 11), cumulative * 100, marker="o", color="#DD8452")
ax2.set_xlabel("Número de componentes")
ax2.set_ylabel("Varianza explicada acumulada (%)")
ax2.set_title("Varianza acumulada")
ax2.set_xticks(range(1, 11))
ax2.axhline(y=80, color="gray", linestyle="--", alpha=0.5, label="80%")
ax2.legend()

plt.suptitle("Varianza explicada por TruncatedSVD (hasta 10 componentes)", fontsize=13)
plt.tight_layout()
plt.savefig("p1_5_pca_varianza.png", dpi=150)
plt.show()

print("Varianza explicada por componente:")
for i, (v, c) in enumerate(zip(explained, cumulative), 1):
    print(f"  SVD{i}: {v*100:.2f}%  (acumulada: {c*100:.2f}%)")

**Análisis de la separación PCA:**

Con solo 2 componentes principales la separación entre medios es parcial: se observan regiones de mayor densidad por medio, pero con considerable superposición. Esto tiene sentido porque 2 componentes capturan una fracción pequeña de la varianza total del espacio TF-IDF.

La separación observada puede deberse a una combinación de factores:
- **Diferencias temáticas:** cada medio cubre ciertos temas con mayor frecuencia, lo que introduce vocabulario diferenciado.
- **Diferencias de estilo editorial:** uso distinto de adjetivos, longitud de oraciones, registro formal/informal.
- **Pistas explícitas:** si el texto contiene menciones directas al nombre del medio (e.g., "Reuters reported"), estas generarían separación artificial que no refleja diferencias semánticas reales. La limpieza del dateline mitiga esto parcialmente.

# Parte 2: Entrenamiento y Evaluación de Modelos

## 1. Multinomial Naive Bayes

**¿Cómo funciona?**
Naive Bayes aplica el teorema de Bayes para estimar la probabilidad de que un documento pertenezca a cada clase dadas sus palabras: P(clase | palabras) ∝ P(clase) × ∏ P(palabra | clase). El supuesto "naive" es que las palabras son condicionalmente independientes dada la clase — simplificación fuerte pero que funciona bien en la práctica para texto.

**Multinomial** porque asume que los conteos de palabras siguen una distribución multinomial (adecuada para BoW/TF-IDF).

Para clasificar: dado un artículo nuevo, calcula la probabilidad para cada medio y elige el de mayor probabilidad.

In [ ]:
# Usar la representación TF-IDF con stop_words e IDF como features
# (ya computadas arriba: X_train_tfidf, X_test_tfidf)
# Importante: fit_transform fue aplicado sobre X_train, transform sobre X_test
# para no filtrar información del test al vectorizador.

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred_nb)
print(f"Accuracy en test: {acc:.4f} ({acc*100:.2f}%)")
print()
print("Reporte completo (precision, recall, F1 por medio):")
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_nb,
    display_labels=nb_model.classes_,
    ax=ax,
    colorbar=False,
    cmap="Blues"
)
ax.set_title("Matriz de confusión — Multinomial Naive Bayes")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("p2_1_confusion_matrix.png", dpi=150)
plt.show()

**Relación entre métricas y la matriz de confusión:**

- **Precision** para un medio = TP / (TP + FP): de todas las predicciones que el modelo hizo como ese medio, ¿cuántas eran correctas? Corresponde a la columna del medio en la matriz (TP en la diagonal, FP en el resto de la columna).
- **Recall** para un medio = TP / (TP + FN): de todos los artículos que realmente son de ese medio, ¿cuántos detectó el modelo? Corresponde a la fila del medio en la matriz.

**Problema de mirar solo accuracy:**
Si un medio tiene el 80% de los artículos, un modelo que siempre prediga ese medio tendría 80% de accuracy sin aprender nada. Con mayor desbalance, el accuracy se vuelve aún más engañoso. Por eso es importante reportar precision y recall por clase.

## 2. Validación cruzada y búsqueda de hiperparámetros

**¿Qué es cross-validation?**
En lugar de un único split train/validation, se divide el conjunto de entrenamiento en K partes (folds). Se entrena K veces: en cada iteración, K-1 folds son entrenamiento y 1 fold es validación. Se promedian los K resultados, obteniendo una estimación más robusta y con menor varianza que un único split.

**GridSearchCV** automatiza este proceso para una grilla de combinaciones de hiperparámetros, eligiendo la combinación con mejor métrica promedio en cross-validation.

In [ ]:
# Grilla de hiperparámetros para Multinomial Naive Bayes
# alpha: parámetro de suavizado de Laplace (evita probabilidades cero)
# Valores más altos = más suavizado = modelo más simple
param_grid = {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]}

grid_search = GridSearchCV(
    MultinomialNB(),
    param_grid,
    cv=5,               # 5-fold cross-validation
    scoring="accuracy",
    return_train_score=False,
    n_jobs=-1           # usar todos los núcleos disponibles
)

t0 = time()
grid_search.fit(X_train_tfidf, y_train)
print(f"GridSearchCV completado en {time()-t0:.1f}s")
print(f"Mejor alpha: {grid_search.best_params_['alpha']}")
print(f"Mejor accuracy (CV): {grid_search.best_score_:.4f}")

In [ ]:
# Visualización: violin plot de accuracy por valor de alpha
cv_results = pd.DataFrame(grid_search.cv_results_)

# Extraer accuracy de cada fold para cada alpha
# GridSearchCV guarda split0_test_score, split1_test_score, etc.
split_cols = [c for c in cv_results.columns if c.startswith("split") and c.endswith("test_score")]

rows = []
for _, row in cv_results.iterrows():
    alpha = row["param_alpha"]
    for col in split_cols:
        rows.append({"alpha": str(alpha), "accuracy": row[col]})

df_violin = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
alphas_ordered = [str(a) for a in param_grid["alpha"]]
sns.violinplot(
    data=df_violin, x="alpha", y="accuracy",
    order=alphas_ordered, ax=ax,
    palette="muted", inner="point"
)
ax.set_xlabel("alpha (parámetro de suavizado de Laplace)")
ax.set_ylabel("Accuracy en CV")
ax.set_title("Distribución de accuracy en cross-validation por valor de alpha (Multinomial NB)")

# Marcar el mejor
best_alpha = str(grid_search.best_params_["alpha"])
best_x = alphas_ordered.index(best_alpha)
ax.axvline(best_x, color="red", linestyle="--", alpha=0.7, label=f"Mejor alpha={best_alpha}")
ax.legend()

plt.tight_layout()
plt.savefig("p2_2_violin.png", dpi=150)
plt.show()

## 3. Entrenamiento final con el mejor modelo

Se reentrena el Naive Bayes con el mejor `alpha` encontrado, esta vez usando **todo el conjunto de entrenamiento** (sin apartar datos para validación). Esto maximiza la cantidad de datos de entrenamiento para el modelo final.

In [ ]:
# Reentrenar con el mejor alpha sobre todo el train
best_alpha = grid_search.best_params_["alpha"]
print(f"Usando alpha={best_alpha}")

best_nb = MultinomialNB(alpha=best_alpha)
best_nb.fit(X_train_tfidf, y_train)

y_pred_best = best_nb.predict(X_test_tfidf)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"\nAccuracy final en test: {acc_best:.4f} ({acc_best*100:.2f}%)")
print()
print("Reporte final:")
print(classification_report(y_test, y_pred_best))

In [ ]:
# Matriz de confusión final
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=best_nb.classes_,
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title(f"Matriz de confusión — Mejor NB (alpha={best_alpha})")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("p2_3_confusion_final.png", dpi=150)
plt.show()

**Limitaciones de BoW/TF-IDF:**

1. **Sin orden:** "perro muerde hombre" y "hombre muerde perro" tienen el mismo vector. Se pierde la sintaxis y el orden de las palabras.
2. **Sin semántica:** palabras sinónimas ("coche" y "automóvil") tienen columnas separadas y el modelo no sabe que son similares.
3. **Alta dimensionalidad:** el vocabulario puede tener decenas de miles de términos, muchos irrelevantes o muy raros.
4. **Vocabulario fijo:** palabras no vistas en el train son ignoradas en el test.
5. **Sin contexto:** la misma palabra tiene el mismo peso sin importar el contexto en que aparezca.

## 4. Modelo alternativo: Regresión Logística

**¿Cómo funciona?**
La Regresión Logística es un clasificador lineal que aprende un peso para cada feature (cada término del vocabulario). Para cada clase, calcula un score lineal (suma ponderada de los TF-IDF) y aplica la función softmax para convertirlo en probabilidades. El modelo se entrena minimizando la pérdida de entropía cruzada.

A diferencia de Naive Bayes, **no asume independencia entre features** — puede capturar correlaciones entre palabras. Suele ser más potente pero también más lento de entrenar.

El parámetro `C` controla la regularización: valores menores de C = mayor regularización = modelo más simple.

In [ ]:
# Entrenar Regresión Logística
# max_iter=1000 para asegurar convergencia con datos de alta dimensión
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1)
t0 = time()
lr_model.fit(X_train_tfidf, y_train)
print(f"Entrenamiento completado en {time()-t0:.1f}s")

y_pred_lr = lr_model.predict(X_test_tfidf)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"\nAccuracy Regresión Logística: {acc_lr:.4f} ({acc_lr*100:.2f}%)")
print(f"Accuracy Naive Bayes (mejor): {acc_best:.4f} ({acc_best*100:.2f}%)")
print()
print("Reporte Regresión Logística:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Matriz de confusión Regresión Logística
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=best_nb.classes_,
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title(f"Naive Bayes (alpha={best_alpha})\nAcc={acc_best:.3f}")
axes[0].tick_params(axis='x', rotation=20)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=lr_model.classes_,
    ax=axes[1], colorbar=False, cmap="Oranges"
)
axes[1].set_title(f"Regresión Logística (C=1.0)\nAcc={acc_lr:.3f}")
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle("Comparación de modelos — Matrices de confusión", fontsize=13)
plt.tight_layout()
plt.savefig("p2_4_comparacion.png", dpi=150)
plt.show()

## 5. Cambio de medio de prensa

Se evalúa el problema con una combinación diferente de tres medios. Se elige al menos un medio con distinto volumen de artículos para analizar el impacto del desbalance de clases.

In [ ]:
# Ver la distribución completa de publicaciones para elegir alternativas
pub_counts = df["publication"].value_counts()
print("Distribución de artículos por publicación:")
print(pub_counts.head(10))

In [ ]:
# Seleccionar una combinación diferente: top 1 + top 5 + top 10
# para introducir desbalance entre medios
alt_pubs = pub_counts.head(10).index.tolist()
# Elegimos posiciones 0, 4, 9 (índices 0-based) — desbalanceados intencionalmente
selected = [alt_pubs[0], alt_pubs[4], alt_pubs[9]]
print("Medios seleccionados para el experimento alternativo:", selected)

df_alt = df[df["publication"].isin(selected)].copy()
df_alt["CleanText"] = clean_text(df_alt, "title", "article")

print("\nDistribución de artículos:")
print(df_alt["publication"].value_counts())
print()

# Visualizar desbalance
fig, ax = plt.subplots(figsize=(7, 4))
df_alt["publication"].value_counts().plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Distribución de artículos — combinación alternativa")
ax.set_xlabel("")
ax.set_ylabel("Cantidad de artículos")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("p2_5_balance.png", dpi=150)
plt.show()

In [ ]:
# Split y vectorización para la combinación alternativa
X_alt = df_alt["CleanText"]
y_alt = df_alt["publication"]

X_tr_alt, X_te_alt, y_tr_alt, y_te_alt = train_test_split(
    X_alt, y_alt, test_size=0.3, stratify=y_alt, random_state=42
)

vec_alt = TfidfVectorizer(stop_words="english", use_idf=True)
X_tr_alt_tfidf = vec_alt.fit_transform(X_tr_alt)
X_te_alt_tfidf = vec_alt.transform(X_te_alt)

# Entrenar NB con mejor alpha
nb_alt = MultinomialNB(alpha=best_alpha)
nb_alt.fit(X_tr_alt_tfidf, y_tr_alt)
y_pred_alt = nb_alt.predict(X_te_alt_tfidf)

acc_alt = accuracy_score(y_te_alt, y_pred_alt)
print(f"Accuracy con combinación alternativa: {acc_alt:.4f}")
print()
print(classification_report(y_te_alt, y_pred_alt))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te_alt, y_pred_alt,
    ax=ax, colorbar=False, cmap="Purples"
)
ax.set_title("Matriz de confusión — Combinación alternativa")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("p2_5_confusion_alt.png", dpi=150)
plt.show()

**Análisis del desbalance:**

Con medios de muy distinto volumen, el modelo tiende a favorecer la clase mayoritaria: tiene más ejemplos para aprender de ella y la predice más frecuentemente. Esto se refleja en:
- Recall alto para la clase mayoritaria (el modelo "recuerda" bien los artículos del medio grande).
- Recall bajo para clases minoritarias (muchos artículos de medios pequeños son clasificados incorrectamente como el medio grande).

**Técnicas para mitigar el desbalance:**
- **Oversampling (sobremuestreo):** duplicar o generar sintéticamente ejemplos de las clases minoritarias. SMOTE (Synthetic Minority Over-sampling Technique) crea nuevos ejemplos interpolando entre ejemplos existentes de la clase minoritaria.
- **Undersampling (submuestreo):** reducir aleatoriamente los ejemplos de la clase mayoritaria para nivelar las clases. Tiene el costo de descartar datos que podrían ser informativos.
- **Pesos de clase:** scikit-learn permite usar `class_weight='balanced'` en muchos modelos, lo que ajusta el costo de los errores inversamente proporcional a la frecuencia de cada clase.

## 6. Técnica alternativa de extracción de features: Word Embeddings

**¿Qué son los word embeddings?**

Los word embeddings (e.g., Word2Vec, GloVe, FastText, o los generados por modelos como BERT) representan cada palabra como un **vector denso** de baja dimensión (típicamente 100–300 dimensiones), aprendido de grandes corpus de texto.

A diferencia de BoW/TF-IDF, los embeddings capturan **similitud semántica**: palabras con significados parecidos tienen vectores cercanos en el espacio. Por ejemplo, los vectores de "rey" y "reina" estarían próximos, y la operación `rey - hombre + mujer ≈ reina` es posible en el espacio de embeddings.

**¿Cómo se usaría para clasificar texto?**

1. Se obtiene el embedding de cada palabra del artículo.
2. Se promedian los embeddings de todas las palabras para obtener un vector representativo del documento (o se usan técnicas más sofisticadas como embeddings de oraciones con modelos tipo Sentence-BERT).
3. Ese vector denso (e.g., de 300 dimensiones) se usa como feature para el clasificador.

**Diferencias esperadas respecto a TF-IDF:**

- **Mejor generalización semántica:** el modelo reconocería que "automóvil" y "coche" son conceptos similares, algo imposible con TF-IDF.
- **Dimensión fija y densa:** el vector de documento tiene siempre la misma dimensión (e.g., 300), independientemente del vocabulario. No hay problema de sparsidad.
- **Manejo de palabras fuera del vocabulario:** modelos como FastText representan palabras nuevas usando subpalabras (n-gramas de caracteres).
- **Contexto:** modelos modernos como BERT generan embeddings contextuales — la misma palabra tiene distinto vector según el contexto en que aparece.

Sin embargo, los embeddings más simples (promedio de Word2Vec) no necesariamente superan a TF-IDF en tareas de clasificación de texto si el vocabulario del dominio es específico y las diferencias entre clases son principalmente léxicas (distintas palabras) en lugar de semánticas.

## 7. Opcional: Clasificación a nivel de título

In [ ]:
# Clasificación usando solo el título del artículo
# Reutilizamos los mismos top 3 medios originales

df_titles = df_top_3.copy()
df_titles["CleanTitle"] = (
    df_titles["title"].fillna("").str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True).str.strip()
)

X_title = df_titles["CleanTitle"]
y_title = df_titles["publication"]

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_title, y_title, test_size=0.3, stratify=y_title, random_state=42
)

vec_title = TfidfVectorizer(stop_words="english", use_idf=True)
X_tr_t_tfidf = vec_title.fit_transform(X_tr_t)
X_te_t_tfidf = vec_title.transform(X_te_t)

nb_title = MultinomialNB(alpha=best_alpha)
nb_title.fit(X_tr_t_tfidf, y_tr_t)
y_pred_title = nb_title.predict(X_te_t_tfidf)

acc_title = accuracy_score(y_te_t, y_pred_title)
print(f"Accuracy clasificando por TÍTULO: {acc_title:.4f}")
print(f"Accuracy clasificando por ARTÍCULO: {acc_best:.4f}")
print()
print(classification_report(y_te_t, y_pred_title))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te_t, y_pred_title,
    ax=ax, colorbar=False, cmap="Greens"
)
ax.set_title(f"Clasificación por título — Accuracy={acc_title:.3f}")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("p2_7_confusion_title.png", dpi=150)
plt.show()